In [ ]:
!pip install ipywidgets
!pip install ipympl
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ----------------------------
# Data
# ----------------------------
n = 16
rng = np.random.default_rng(42)
heights = rng.choice(np.arange(1, 101), size=n, replace=False)  # all distinct

selected = {"i": 7}
n_neighbours = 2

# ----------------------------
# UI elements
# ----------------------------
btn_left = widgets.Button(description="◀", layout=widgets.Layout(width="60px"))
btn_right = widgets.Button(description="▶", layout=widgets.Layout(width="60px"))
label = widgets.HTML()
out = widgets.Output()

def update_label():
    i = selected["i"]
    label.value = (
        f"<b>Selected bar:</b> {i+1}/{n} &nbsp;&nbsp;"
        f"<b>Height:</b> {heights[i]}"
    )

# ----------------------------
# Drawing
# ----------------------------
def draw():
    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(12, 5))

        x = np.arange(n)
        bars = ax.bar(x, heights)

        sel = selected["i"]

        for j, b in enumerate(bars):
            if j == sel:
                b.set_hatch("//")
            if j <sel+n_neighbours and j>sel-n_neighbours:
                b.set_linewidth(3.0)
                b.set_alpha(1.0)
                b.set_facecolor('#F27761')
            else:
                b.set_linewidth(1.0)
                b.set_hatch(None)
                b.set_alpha(0.5)
            

        # ----- arrow + value annotation -----
        h = heights[sel]
        ax.annotate(
            "▲",
            xy=(sel, h),
            xytext=(sel, h + 6),
            ha="center",
            va="bottom",
            fontsize=16,
            fontweight="bold",
        )
        ax.text(
            sel,
            h + 0.4,
            f"{h}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )

        ax.set_title("Click a bar or use ◀ / ▶ to move the selection")
        ax.set_xticks(x)
        ax.set_xticklabels([str(i + 1) for i in range(n)])
        ax.set_xlim(-0.75, n - 0.25)
        ax.set_ylim(0, max(heights) * 1.35)

        # ----- mouse click handler -----
        def on_click(event):
            if event.inaxes != ax or event.xdata is None:
                return
            idx = int(round(event.xdata))
            if 0 <= idx < n:
                selected["i"] = idx
                update_label()
                draw()

        fig.canvas.mpl_connect("button_press_event", on_click)
        plt.show()

# ----------------------------
# Button callbacks
# ----------------------------
def on_left(_):
    selected["i"] = (selected["i"] - 1) % n
    update_label()
    draw()

def on_right(_):
    selected["i"] = (selected["i"] + 1) % n
    update_label()
    draw()

btn_left.on_click(on_left)
btn_right.on_click(on_right)

# ----------------------------
# Initial render
# ----------------------------
update_label()
display(widgets.HBox([btn_left, btn_right, label]))
display(out)
draw()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# =========================
# Grid config (match the image)
# =========================
H, W = 5, 10   # rows, cols

# 🏠 locations (row, col) — from the image
houses = [
    (0, 8),
    (1, 2),
    (3, 1),
    (4, 6),
]

hospitals = [(0,4),(3,9)]

# Manhattan distance
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def in_bounds(r, c):
    return 0 <= r < H and 0 <= c < W

def neighbors4(r, c):
    for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
        rr, cc = r + dr, c + dc
        if in_bounds(rr, cc):
            yield (rr, cc)

def neighbors8(r, c):
    for dr, dc in [(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (1,1), (1,-1), (-1,1)]:
        rr, cc = r + dr, c + dc
        if in_bounds(rr, cc):
            yield (rr, cc)

def sum_two_nearest_houses(cell):
    dists = sorted(manhattan(cell, h) for h in houses)
    return dists[0] + dists[1] 

def two_nearest_houses(cell):
    """Return the two nearest house cells to `cell`."""
    return sorted(houses, key=lambda h: manhattan(cell, h))[:2]

def hospital_cost(hosp):
    """Sum of distances from hosp to its two nearest houses."""
    if hosp is None:
        return 0
    hs = two_nearest_houses(hosp)
    return sum(manhattan(hosp, h) for h in hs)

def total_cost():
    """Total cost = cost(hosp1) + cost(hosp2)."""
    return hospital_cost(state["hosp1"]) + hospital_cost(state["hosp2"])



# =========================
# State
# =========================
state = {
    "clicks": 0,
    "hosp1": None,
    "hosp2": None,
    "highlight1": set(),
    "highlight2": set(),
}

# =========================
# Drawing setup
# =========================
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_ylim(H, -0.8)  # give some space above the grid
ax.set_aspect("equal")
ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.set_xticks(np.arange(W + 1))
ax.set_yticks(np.arange(H + 1))
ax.grid(True, linewidth=2, color="white")
ax.set_facecolor("black")
ax.set_title("Odd click: place Hospital 1 | Even click: place Hospital 2", color="white")

_art = {"rects": [], "texts": []}

def draw_manhattan_path(a, b, color="#39d353", order="HV"):
    """
    Draw a Manhattan (L-shaped) path on the grid from cell a to cell b.
    a, b are (row, col). Path goes through cell centers.
    order:
      - "HV": horizontal then vertical
      - "VH": vertical then horizontal
    """
    (ra, ca), (rb, cb) = a, b
    xa, ya = ca + 0.5, ra + 0.5
    xb, yb = cb + 0.5, rb + 0.5

    if order == "HV":
        xm, ym = xb, ya  # go horizontally to target column, then vertically
    else:
        xm, ym = xa, yb  # go vertically to target row, then horizontally

    line = ax.plot(
        [xa, xm, xb],
        [ya, ym, yb],
        color='#1CB0FF',
        linewidth=5,
        alpha=0.9,
        zorder=2
    )[0]
    _art["rects"].append(line)
        

def clear_art():
    for x in _art["rects"] + _art["texts"]:
        try: x.remove()
        except: pass
    _art["rects"].clear()
    _art["texts"].clear()

def fill_cell(rc, color, alpha=1.0):
    if rc not in houses:
        r, c = rc
        rect = Rectangle((c, r), 1, 1, facecolor=color, edgecolor="none", alpha=alpha)
        ax.add_patch(rect)
        _art["rects"].append(rect)

from matplotlib.patches import Polygon, Circle, Rectangle

def draw_house(rc):
    """Draw a white 'house' icon similar to the screenshot."""
    r, c = rc
    x0, y0 = c, r

    # roof (triangle)
    roof = Polygon(
        [(x0+0.20, y0+0.55), (x0+0.50, y0+0.25), (x0+0.80, y0+0.55)],
        closed=True, facecolor="white", edgecolor="none", zorder=3
    )
    ax.add_patch(roof); _art["rects"].append(roof)

    # body (rectangle)
    body = Rectangle(
        (x0+0.28, y0+0.55), 0.44, 0.30,
        facecolor="white", edgecolor="none", zorder=3
    )
    ax.add_patch(body); _art["rects"].append(body)

    # door (black cut-out)
    door = Rectangle(
        (x0+0.42, y0+0.67), 0.16, 0.18,
        facecolor="black", edgecolor="none", zorder=4
    )
    ax.add_patch(door); _art["rects"].append(door)

def draw_hospital(rc):
    """Draw a white 'hospital' icon with a red cross."""
    r, c = rc
    x0, y0 = c, r

    # main white rounded-ish block (use rect; looks clean on projector)
    block = Rectangle(
        (x0+0.28, y0+0.18), 0.44, 0.70,
        facecolor="white", edgecolor="none", zorder=3
    )
    ax.add_patch(block); _art["rects"].append(block)

    # bottom entrance cut-out (black)
    door = Rectangle(
        (x0+0.42, y0+0.68), 0.16, 0.20,
        facecolor="black", edgecolor="none", zorder=4
    )
    ax.add_patch(door); _art["rects"].append(door)

    # black circle behind cross (like screenshot)
    circ = Circle((x0+0.50, y0+0.34), 0.16, facecolor="black", edgecolor="none", zorder=4)
    ax.add_patch(circ); _art["rects"].append(circ)

    # red cross (two rectangles)
    cross_v = Rectangle((x0+0.48, y0+0.24), 0.04, 0.20, facecolor="#ff2a2a", edgecolor="none", zorder=5)
    cross_h = Rectangle((x0+0.40, y0+0.32), 0.20, 0.04, facecolor="#ff2a2a", edgecolor="none", zorder=5)
    ax.add_patch(cross_v); _art["rects"].append(cross_v)
    ax.add_patch(cross_h); _art["rects"].append(cross_h)

def draw_icon(rc, kind):
    """kind in {'house','hospital'}"""
    if kind == "house":
        draw_house(rc)
    elif kind == "hospital":
        draw_hospital(rc)


def draw_number(rc, val):
    if rc not in houses:
        r, c = rc
        t = ax.text(c + 0.5, r + 0.5, str(val),
                    ha="center", va="center",
                    fontsize=20, fontweight="bold",
                    color="#39d353", zorder=10)
        _art["texts"].append(t)

def init():
    clear_art()

    for h in houses:
        draw_icon(h, "house")
        
    for hos in hospitals:
        draw_icon(hos, "hospital")
        
    fig.canvas.draw_idle()

def choose_order(a, b, occupied):
    (ra, ca), (rb, cb) = a, b
    # midpoints for HV and VH
    hv_mid = (ra, cb)
    vh_mid = (rb, ca)

    if hv_mid in occupied and vh_mid not in occupied:
        return "VH"
    if vh_mid in occupied and hv_mid not in occupied:
        return "HV"
    return "HV"  # default

def redraw():
    clear_art()

    # highlight cells
    for rc in state["highlight1"] | state["highlight2"]:
        fill_cell(rc, "white", alpha=0.5)

    for h in houses:
        draw_icon(h, "house")
    
    if state["hosp1"] is not None:
        draw_icon(state["hosp1"], "hospital")
    if state["hosp2"] is not None:
        draw_icon(state["hosp2"], "hospital")
    
    # connections: hospital -> nearest two houses
    occupied = set(houses)
    if state["hosp1"] is not None: occupied.add(state["hosp1"])
    if state["hosp2"] is not None: occupied.add(state["hosp2"])
    for hosp in [state["hosp1"], state["hosp2"]]:
        if hosp is not None:
            for h in two_nearest_houses(hosp):
                draw_manhattan_path(hosp, h, order=choose_order(hosp, h, occupied))

    # distance numbers
    for rc in state["highlight1"] | state["highlight2"]:
        draw_number(rc, sum_two_nearest_houses(rc))

    # --- total cost display at the top ---
    tc = total_cost()
    t = ax.text(
        W/2, -0.25, f"Total cost = {tc}",
        ha="center", va="bottom",
        fontsize=16, fontweight="bold",
        color="#39d353",
        zorder=10
    )
    _art["texts"].append(t)


    fig.canvas.draw_idle()

# =========================
# Click handling
# =========================
def cell_from_event(event):
    if event.inaxes != ax or event.xdata is None or event.ydata is None:
        return None
    c = int(event.xdata)
    r = int(event.ydata)
    if not in_bounds(r, c):
        return None
    return (r, c)

def place_hospital(hosp_key, hl_key, rc):
    state[hosp_key] = rc
    # state[hl_key] = {rc} | set(neighbors4(*rc))
    state[hl_key] = {rc} | set(neighbors8(*rc))

def on_click(event):
    rc = cell_from_event(event)
    if rc is None:
        return

    state["clicks"] += 1
    if state["clicks"] % 2 == 1:
        place_hospital("hosp1", "highlight1", rc)
    else:
        place_hospital("hosp2", "highlight2", rc)

    redraw()

fig.canvas.mpl_connect("button_press_event", on_click)

# initial draw
init()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# =========================
# Grid config (match the image)
# =========================
H, W = 5, 10   # rows, cols

# 🏠 locations (row, col) — from the image
houses = [
    (1, 2),
    (3, 1),
    (4, 6),
    (0, 8),
]

hospitals = [(0,4),(3,9)]

# Manhattan distance
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def in_bounds(r, c):
    return 0 <= r < H and 0 <= c < W

def neighbors4(r, c):
    for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
        rr, cc = r + dr, c + dc
        if in_bounds(rr, cc):
            yield (rr, cc)

def neighbors8(r, c):
    for dr, dc in [(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (1,1), (1,-1), (-1,1)]:
        rr, cc = r + dr, c + dc
        if in_bounds(rr, cc):
            yield (rr, cc)

def sum_two_nearest_houses(cell,i):
    # dists = sorted(manhattan(cell, h) for h in houses)
    # return dists[0] + dists[1] 
    if i==0:
        return sum([manhattan(cell, h) for h in houses[:2]])
    else: return sum([manhattan(cell, h) for h in houses[2:]])

def two_nearest_houses(cell,i):
    """Return the two nearest house cells to `cell`."""
    # return sorted(houses, key=lambda h: manhattan(cell, h))[:2]
    if i==0:
        return houses[:2]
    else: return houses[2:]

def hospital_cost(hosp):
    """Sum of distances from hosp to its two nearest houses."""
    if hosp is None:
        return 0
    hs = two_nearest_houses(hosp)
    return sum(manhattan(hosp, h) for h in hs)

def total_cost():
    """Total cost = cost(hosp1) + cost(hosp2)."""
    return sum_two_nearest_houses(state["hosp1"],0) + sum_two_nearest_houses(state["hosp2"],1)



# =========================
# State
# =========================
state = {
    "clicks": 0,
    "hosp1": None,
    "hosp2": None,
    "highlight1": set(),
    "highlight2": set(),
}

# =========================
# Drawing setup
# =========================
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_ylim(H, -0.8)  # give some space above the grid
ax.set_aspect("equal")
ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.set_xticks(np.arange(W + 1))
ax.set_yticks(np.arange(H + 1))
ax.grid(True, linewidth=2, color="white")
ax.set_facecolor("black")
ax.set_title("Odd click: place Hospital 1 | Even click: place Hospital 2", color="white")

_art = {"rects": [], "texts": []}

def draw_manhattan_path(a, b, color="#39d353", order="HV"):
    """
    Draw a Manhattan (L-shaped) path on the grid from cell a to cell b.
    a, b are (row, col). Path goes through cell centers.
    order:
      - "HV": horizontal then vertical
      - "VH": vertical then horizontal
    """
    (ra, ca), (rb, cb) = a, b
    xa, ya = ca + 0.5, ra + 0.5
    xb, yb = cb + 0.5, rb + 0.5

    if order == "HV":
        xm, ym = xb, ya  # go horizontally to target column, then vertically
    else:
        xm, ym = xa, yb  # go vertically to target row, then horizontally

    line = ax.plot(
        [xa, xm, xb],
        [ya, ym, yb],
        color='#1CB0FF',
        linewidth=5,
        alpha=0.9,
        zorder=2
    )[0]
    _art["rects"].append(line)
        

def clear_art():
    for x in _art["rects"] + _art["texts"]:
        try: x.remove()
        except: pass
    _art["rects"].clear()
    _art["texts"].clear()

def fill_cell(rc, color, alpha=0.3):
    if rc not in houses:
        r, c = rc
        rect = Rectangle((c, r), 1, 1, facecolor=color, edgecolor="none", alpha=alpha)
        ax.add_patch(rect)
        _art["rects"].append(rect)

from matplotlib.patches import Polygon, Circle, Rectangle

def draw_house(rc):
    """Draw a white 'house' icon similar to the screenshot."""
    r, c = rc
    x0, y0 = c, r

    # roof (triangle)
    roof = Polygon(
        [(x0+0.20, y0+0.55), (x0+0.50, y0+0.25), (x0+0.80, y0+0.55)],
        closed=True, facecolor="white", edgecolor="none", zorder=3
    )
    ax.add_patch(roof); _art["rects"].append(roof)

    # body (rectangle)
    body = Rectangle(
        (x0+0.28, y0+0.55), 0.44, 0.30,
        facecolor="white", edgecolor="none", zorder=3
    )
    ax.add_patch(body); _art["rects"].append(body)

    # door (black cut-out)
    door = Rectangle(
        (x0+0.42, y0+0.67), 0.16, 0.18,
        facecolor="black", edgecolor="none", zorder=4
    )
    ax.add_patch(door); _art["rects"].append(door)

def draw_hospital(rc):
    """Draw a white 'hospital' icon with a red cross."""
    r, c = rc
    x0, y0 = c, r

    # main white rounded-ish block (use rect; looks clean on projector)
    block = Rectangle(
        (x0+0.28, y0+0.18), 0.44, 0.70,
        facecolor="white", edgecolor="none", zorder=3
    )
    ax.add_patch(block); _art["rects"].append(block)

    # bottom entrance cut-out (black)
    door = Rectangle(
        (x0+0.42, y0+0.68), 0.16, 0.20,
        facecolor="black", edgecolor="none", zorder=4
    )
    ax.add_patch(door); _art["rects"].append(door)

    # black circle behind cross (like screenshot)
    circ = Circle((x0+0.50, y0+0.34), 0.16, facecolor="black", edgecolor="none", zorder=4)
    ax.add_patch(circ); _art["rects"].append(circ)

    # red cross (two rectangles)
    cross_v = Rectangle((x0+0.48, y0+0.24), 0.04, 0.20, facecolor="#ff2a2a", edgecolor="none", zorder=5)
    cross_h = Rectangle((x0+0.40, y0+0.32), 0.20, 0.04, facecolor="#ff2a2a", edgecolor="none", zorder=5)
    ax.add_patch(cross_v); _art["rects"].append(cross_v)
    ax.add_patch(cross_h); _art["rects"].append(cross_h)

def draw_icon(rc, kind):
    """kind in {'house','hospital'}"""
    if kind == "house":
        draw_house(rc)
    elif kind == "hospital":
        draw_hospital(rc)


def draw_number(rc, val):
    if rc not in houses:
        r, c = rc
        t = ax.text(c + 0.5, r + 0.5, str(val),
                    ha="center", va="center",
                    fontsize=20, fontweight="bold",
                    color="#39d353", zorder=10)
        _art["texts"].append(t)

def init():
    clear_art()

    for h in houses:
        draw_icon(h, "house")
        
    for hos in hospitals:
        draw_icon(hos, "hospital")
        
    fig.canvas.draw_idle()

def choose_order(a, b, occupied):
    (ra, ca), (rb, cb) = a, b
    # midpoints for HV and VH
    hv_mid = (ra, cb)
    vh_mid = (rb, ca)

    if hv_mid in occupied and vh_mid not in occupied:
        return "VH"
    if vh_mid in occupied and hv_mid not in occupied:
        return "HV"
    return "HV"  # default

def redraw():
    clear_art()

    # highlight cells
    for rc in state["highlight1"] | state["highlight2"]:
        fill_cell(rc, "w", alpha=0.5)

    for h in houses:
        draw_icon(h, "house")
    
    if state["hosp1"] is not None:
        draw_icon(state["hosp1"], "hospital")
    if state["hosp2"] is not None:
        draw_icon(state["hosp2"], "hospital")
    
    # connections: hospital -> nearest two houses
    occupied = set(houses)
    if state["hosp1"] is not None: occupied.add(state["hosp1"])
    if state["hosp2"] is not None: occupied.add(state["hosp2"])
    for hosp in [state["hosp1"]]:
        if hosp is not None:
            for h in two_nearest_houses(hosp,0):
                draw_manhattan_path(hosp, h, order=choose_order(hosp, h, occupied))
    for hosp in [state["hosp2"]]:
        if hosp is not None:
            for h in two_nearest_houses(hosp,1):
                draw_manhattan_path(hosp, h, order=choose_order(hosp, h, occupied))

    # distance numbers
    for rc in state["highlight1"]:
        draw_number(rc, sum_two_nearest_houses(rc, 0))
    for rc in state["highlight2"]:
        draw_number(rc, sum_two_nearest_houses(rc, 1))

    # --- total cost display at the top ---
    tc = total_cost()
    t = ax.text(
        W/2, -0.25, f"Total cost = {tc}",
        ha="center", va="bottom",
        fontsize=16, fontweight="bold",
        color="#39d353",
        zorder=10
    )
    _art["texts"].append(t)

    fig.canvas.draw_idle()

# =========================
# Click handling
# =========================
def cell_from_event(event):
    if event.inaxes != ax or event.xdata is None or event.ydata is None:
        return None
    c = int(event.xdata)
    r = int(event.ydata)
    if not in_bounds(r, c):
        return None
    return (r, c)

def place_hospital(hosp_key, hl_key, rc):
    state[hosp_key] = rc
    # state[hl_key] = {rc} | set(neighbors4(*rc))
    state[hl_key] = {rc} | set(neighbors8(*rc))

def on_click(event):
    rc = cell_from_event(event)
    if rc is None:
        return

    state["clicks"] += 1
    if state["clicks"] % 2 == 1:
        place_hospital("hosp1", "highlight1", rc)
    else:
        place_hospital("hosp2", "highlight2", rc)

    redraw()

fig.canvas.mpl_connect("button_press_event", on_click)

# initial draw
init()
plt.show()
